In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 3 と Qiskit SDK

<details>
<summary><b>パッケージバージョン</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降を使用することを推奨します。

```
qiskit[all]~=2.3.0
```
</details>

Qiskit SDK は、量子プログラムの OpenQASM 表現と QuantumCircuit クラスの間で変換するためのツールを提供しています。なお、これらのツールはまだ開発の探索段階にあり、OpenQASM 3 で表現される動的回路機能に対する Qiskit のサポートが拡充されるにつれて、今後も進化し続けます。

> **Note:** この機能はまだ探索段階にあります。そのため、構文や機能が変更される可能性があります。

## OpenQASM 3 プログラムを Qiskit にインポートする
この機能を使用するには、`qiskit_qasm3_import ` パッケージをインストールする必要があります。以下のコマンドでインストールしてください。

In [1]:
import qiskit.qasm3

program = """
    OPENQASM 3.0;
    include "stdgates.inc";

    input float[64] a;
    qubit[3] q;
    bit[2] mid;
    bit[3] out;

    let aliased = q[0:1];

    gate my_gate(a) c, t {
      gphase(a / 2);
      ry(a) c;
      cx c, t;
    }
    gate my_phase(a) c {
      ctrl @ inv @ gphase(a) c;
    }

    my_gate(a * 2) aliased[0], q[{1, 2}][0];
    measure q[0] -> mid[0];
    measure q[1] -> mid[1];

    while (mid == "00") {
      reset q[0];
      reset q[1];
      my_gate(a) q[0], q[1];
      my_phase(a - pi/2) q[1];
      mid[0] = measure q[0];
      mid[1] = measure q[1];
    }

    if (mid[0]) {
      let inner_alias = q[{0, 1}];
      reset inner_alias;
    }

    out = measure q;
"""
circuit = qiskit.qasm3.loads(program)
circuit.draw("mpl")

<Image src="../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg" alt="Output of the previous code cell" />

現在、OpenQASM 3 から Qiskit へインポートするための高レベル関数が 2 つ利用できます。ファイル名を受け取る `load()` と、プログラム自体を文字列として受け取る `loads()` です。

In [2]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

dumps(qc)

'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[2] meas;\nqubit[2] q;\nh q[0];\ncx q[0], q[1];\nbarrier q[0], q[1];\nmeas[0] = measure q[0];\nmeas[1] = measure q[1];\n'

この例では、OpenQASM 3 を使用して量子プログラムを定義し、`loads()` を使って直接 QuantumCircuit に変換します。

In [3]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dump

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

f = open("my_file.txt", "w")
dump(qc, f)
f.close()

![Output of the previous code cell](../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg)

## OpenQASM 3 へエクスポートする
Qiskit コードを OpenQASM 3 にエクスポートするには、文字列としてエクスポートする `dumps()` またはファイルにエクスポートする `dump()` を使用できます。

### `dumps()` を使った例